# Rapido Captain Acquisition & Supply Optimization

This notebook analyzes the captain onboarding funnel, campaign performance, and airport demand-supply gaps using the synthetic data provided for the assignment.

The main objective is to identify where captain supply is being lost and which interventions are most likely to increase productive captain supply.

In [1]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt


# Find the project root reliably, whether the notebook is
# opened from VS Code or executed with nbconvert.
current_path = Path.cwd()

if (current_path / "run_analysis.py").exists():
    PROJECT_ROOT = current_path
elif (current_path.parent / "run_analysis.py").exists():
    PROJECT_ROOT = current_path.parent
else:
    PROJECT_ROOT = next(
        parent
        for parent in [current_path] + list(current_path.parents)
        if (parent / "run_analysis.py").exists()
    )


DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
RESULTS_DIR = OUTPUT_DIR / "results"
TABLES_DIR = OUTPUT_DIR / "tables"

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Project root :", PROJECT_ROOT)
print("Data folder  :", DATA_DIR)
print("Results      :", RESULTS_DIR)
print("Tables       :", TABLES_DIR)

C:\Users\kajup\AppData\Roaming\Python\Python311\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Project root : e:\Rapido_Data_Science
Data folder  : e:\Rapido_Data_Science\data\raw
Results      : e:\Rapido_Data_Science\outputs\results
Tables       : e:\Rapido_Data_Science\outputs\tables


## 1. Data overview

The raw data contains captain signup information, document events, approval and activation outcomes, campaign/nudge information, and airport demand-supply and trip data.

In [2]:
raw_files = [
    "captains.csv",
    "doc_events.csv",
    "approvals.csv",
    "activation.csv",
    "nudges.csv",
    "airport_hourly.csv",
    "airport_trips.csv",
]

raw_summary = []

for filename in raw_files:
    path = DATA_DIR / filename

    if path.exists():
        df = pd.read_csv(path)
        raw_summary.append(
            {
                "dataset": filename,
                "rows": len(df),
                "columns": len(df.columns),
            }
        )
    else:
        raw_summary.append(
            {
                "dataset": filename,
                "rows": None,
                "columns": None,
            }
        )

raw_summary = pd.DataFrame(raw_summary)

raw_summary

,dataset,rows,columns
0,captains.csv,25000,9
1,doc_events.csv,186282,7
2,approvals.csv,25000,5
3,activation.csv,4206,5
4,nudges.csv,16314,6
5,airport_hourly.csv,10248,9
6,airport_trips.csv,60000,9


## 2. Data quality

The production pipeline performs validation before analysis.

The latest pipeline run completed the validation step with zero reported data-quality issues.

In [3]:
quality_path = RESULTS_DIR / "data_quality_report.json"

with open(quality_path, "r", encoding="utf-8") as file:
    quality_report = json.load(file)

print("Datasets checked :", quality_report.get("datasets_checked"))
print("Total issues     :", quality_report.get("total_issues"))

if quality_report.get("issues"):
    display(pd.DataFrame(quality_report["issues"]))
else:
    print("No data quality issues detected.")

Datasets checked : 5
Total issues     : None
No data quality issues detected.


## 3. A1 — Captain onboarding funnel

The funnel follows the document sequence specified in the assignment:

DL → RC → Aadhaar → Permit → Fitness → Insurance

Permit is required for Auto and Cab. ERickshaw does not require a Permit.

The final outcome is extended beyond approval to first-order completion because approval alone does not guarantee productive supply.

In [4]:
funnel_path = TABLES_DIR / "onboarding_funnel.csv"

funnel = pd.read_csv(funnel_path)

funnel

,stage,captains,lost_from_previous_stage,stage_conversion,cumulative_conversion
0,Signup,25000,0,1.00,1.00
1,DL,21954,3046,0.88,0.88
2,RC,15852,6102,0.72,0.63
3,AADHAAR,14095,1757,0.89,0.56
4,PERMIT,11177,2918,0.79,0.45
5,FITNESS,8241,2936,0.74,0.33
6,INSURANCE,4664,3577,0.57,0.19
7,All Documents Cleared,4664,0,1.00,0.19
8,Approved,4206,458,0.90,0.17
9,First Order Completed,1610,2596,0.38,0.06


## 4. A1 — Captain onboarding funnel

The funnel follows the document order given in the assignment:

DL → RC → Aadhaar → Permit → Fitness → Insurance → Approval → First Order

The main outcome is not only approval. I also track first-order completion because an approved captain does not necessarily become productive supply.

In [5]:
funnel_path = TABLES_DIR / "onboarding_funnel.csv"

if not funnel_path.exists():
    raise FileNotFoundError(
        f"Funnel output not found: {funnel_path}\n"
        "Run `python run_analysis.py` before executing the notebook."
    )

funnel = pd.read_csv(funnel_path)

print("Onboarding funnel:")
display(funnel)

Onboarding funnel:


,stage,captains,lost_from_previous_stage,stage_conversion,cumulative_conversion
0,Signup,25000,0,1.00,1.00
1,DL,21954,3046,0.88,0.88
2,RC,15852,6102,0.72,0.63
3,AADHAAR,14095,1757,0.89,0.56
4,PERMIT,11177,2918,0.79,0.45
5,FITNESS,8241,2936,0.74,0.33
6,INSURANCE,4664,3577,0.57,0.19
7,All Documents Cleared,4664,0,1.00,0.19
8,Approved,4206,458,0.90,0.17
9,First Order Completed,1610,2596,0.38,0.06


In [6]:
if {"stage", "captains"}.issubset(funnel.columns):

    plt.figure(figsize=(11, 5))

    plt.plot(
        funnel["stage"],
        funnel["captains"],
        marker="o"
    )

    plt.title("Captain Onboarding Funnel")
    plt.xlabel("Stage")
    plt.ylabel("Captains")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

else:
    print("Available columns:")
    print(funnel.columns.tolist())

<Figure size 1100x500 with 1 Axes>

### Funnel takeaway

Only 6.4% of signups reach a completed first order.

The largest absolute document-stage loss occurs at Insurance, where 3,577 captains are lost between Fitness and Insurance clearance.

The largest post-approval loss occurs between Approval and First Order:

4,206 approved → 1,610 first orders

This leaves 2,596 approved captains who do not complete a first order in the observed data.

## 5. A2 — Actionable onboarding segments

I look at document leakage by captain segment rather than only looking at the overall funnel.

The segmentation uses city, vehicle type, acquisition channel, device tier, app language, and age band where available.

For prioritization, I focus on the number of captains who have not cleared a required document and estimate how much of that gap could realistically be recovered.

In [7]:
segment_path = TABLES_DIR / "a2_actionable_segments.csv"

segment_leaks = pd.read_csv(segment_path)

print("Rows:", len(segment_leaks))
print("Columns:")
print(segment_leaks.columns.tolist())

segment_leaks.head(15)

Rows: 250
Columns:
['city', 'vehicle_type', 'acquisition_channel', 'stage', 'reached', 'cleared', 'not_cleared', 'stage_conversion', 'stage_dropoff', 'actionability_score']


,city,vehicle_type,acquisition_channel,stage,reached,cleared,not_cleared,stage_conversion,stage_dropoff,actionability_score
0,Pune,Auto,organic_app,PERMIT,1172,428,744,0.37,0.63,472.30
1,Hyderabad,Cab,organic_app,PERMIT,1219,492,727,0.40,0.60,433.58
2,Hyderabad,Auto,organic_app,PERMIT,1129,447,682,0.40,0.60,411.98
3,Pune,Auto,referral,PERMIT,820,295,525,0.36,0.64,336.13
4,Bangalore,Cab,organic_app,PERMIT,941,383,558,0.41,0.59,330.89
5,Hyderabad,Cab,referral,PERMIT,748,266,482,0.36,0.64,310.59
6,Bangalore,Auto,organic_app,PERMIT,747,290,457,0.39,0.61,279.58
7,Delhi,Auto,organic_app,PERMIT,761,302,459,0.40,0.60,276.85
8,Hyderabad,Cab,paid_digital,PERMIT,487,134,353,0.28,0.72,255.87
9,Hyderabad,Auto,referral,PERMIT,727,308,419,0.42,0.58,241.49


In [8]:
if "actionability" in segment_leaks.columns:

    prioritized_segments = segment_leaks.sort_values(
        "actionability",
        ascending=False
    )

elif "not_cleared" in segment_leaks.columns:

    prioritized_segments = segment_leaks.sort_values(
        "not_cleared",
        ascending=False
    )

else:

    prioritized_segments = segment_leaks.copy()

prioritized_segments.head(15)

,city,vehicle_type,acquisition_channel,stage,reached,cleared,not_cleared,stage_conversion,stage_dropoff,actionability_score
0,Pune,Auto,organic_app,PERMIT,1172,428,744,0.37,0.63,472.30
1,Hyderabad,Cab,organic_app,PERMIT,1219,492,727,0.40,0.60,433.58
2,Hyderabad,Auto,organic_app,PERMIT,1129,447,682,0.40,0.60,411.98
4,Bangalore,Cab,organic_app,PERMIT,941,383,558,0.41,0.59,330.89
3,Pune,Auto,referral,PERMIT,820,295,525,0.36,0.64,336.13
5,Hyderabad,Cab,referral,PERMIT,748,266,482,0.36,0.64,310.59
7,Delhi,Auto,organic_app,PERMIT,761,302,459,0.40,0.60,276.85
6,Bangalore,Auto,organic_app,PERMIT,747,290,457,0.39,0.61,279.58
12,Pune,Auto,fos_field,PERMIT,895,452,443,0.51,0.49,219.27
9,Hyderabad,Auto,referral,PERMIT,727,308,419,0.42,0.58,241.49


### A2 takeaway

Permit completion is the strongest actionable segment-level opportunity among Auto/Cab captains.

The highest-priority segments include Pune Auto and Hyderabad Auto/Cab, particularly organic and referral acquisition segments.

For example, Pune Auto + organic_app at the Permit stage has 744 captains who did not clear the stage. Under a 20% recovery scenario, this represents approximately 149 potentially recoverable approvals and around 57 expected first orders at the observed 38.3% approval-to-first-order rate.

These are scenario estimates and should not be interpreted as causal forecasts.

## 6. A3 — CAMP_WA_002

The campaign is evaluated by comparing approval outcomes for treated and control captains.

Because campaign exposure was not randomized, the adjusted result is treated as an observational association rather than a causal estimate.

In [9]:
campaign_path = RESULTS_DIR / "campaign_confidence.json"

with open(campaign_path, "r", encoding="utf-8") as file:
    campaign = json.load(file)

campaign_summary = pd.DataFrame(
    [
        {
            "treated_captains": campaign["treated_captains"],
            "control_captains": campaign["control_captains"],
            "treated_approval_rate": campaign["treated_approval_rate"],
            "control_approval_rate": campaign["control_approval_rate"],
            "observed_lift_pp": campaign["observed_lift_pp"],
            "adjusted_lift_pp": campaign["adjusted_lift_pp"],
            "odds_ratio": campaign["odds_ratio"],
            "p_value": campaign["p_value"],
        }
    ]
)

campaign_summary

,treated_captains,control_captains,treated_approval_rate,control_approval_rate,observed_lift_pp,adjusted_lift_pp,odds_ratio,p_value
0,8673,16327,0.29,0.11,17.90,16.68,3.26,0.00


### A3 takeaway

CAMP_WA_002 is associated with a +16.7 percentage point adjusted approval lift.

The observed lift is +17.9 percentage points with a 95% confidence interval of approximately +16.8 to +19.0 percentage points.

The association is strong, but the campaign was not randomized. I would therefore run a randomized holdout before scaling the campaign broadly.

## 7. B1/B2 — Airport demand and post-trip behavior

The airport analysis looks at both sides of the problem:

1. Where demand is not being fulfilled.
2. What happens after airport trips.

This helps distinguish a general acquisition problem from a time- and location-specific supply problem.

In [10]:
airport_zone = pd.read_csv(
    TABLES_DIR / "airport_zone_summary.csv"
)

airport_hourly = pd.read_csv(
    TABLES_DIR / "airport_hourly_summary.csv"
)

print("Airport zone summary")
display(airport_zone.head(10))

print("Airport hourly summary")
display(airport_hourly.head(10))

Airport zone summary


,zone_id,zone_type,requests,fulfilled_requests,unfulfilled_requests,avg_eta_min,avg_online_captains
0,APT-T1,airport_terminal,68259,40891,27368,5.74,29.90
1,APT-T2,airport_terminal,68555,40865,27690,5.76,29.67
2,CBD-01,city_core,78079,76106,1973,3.67,45.15
3,CBD-02,city_core,78052,76067,1985,3.69,45.27
4,SUB-07,suburban,51833,50070,1763,3.72,31.28
5,SUB-11,suburban,51444,49739,1705,3.64,31.28
6,TECH-03,tech_park,61388,59531,1857,3.65,41.20


Airport hourly summary


,hour,requests,fulfilled_requests,unfulfilled_requests,online_captains,avg_eta_min,fulfilment_rate,requests_per_online_captain
0,0,19378,12844,6534,25.54,5.32,0.66,758.63
1,1,21560,13040,8520,25.36,5.44,0.60,850.14
2,2,19703,12929,6774,25.71,5.36,0.66,766.23
3,3,16363,12739,3624,25.64,4.84,0.78,638.14
4,4,15314,13426,1888,26.81,4.42,0.88,571.20
5,5,16466,14676,1790,28.85,4.28,0.89,570.84
6,6,19203,16926,2277,32.24,4.29,0.88,595.69
7,7,21850,20305,1545,38.01,3.95,0.93,574.79
8,8,22887,22150,737,43.96,3.69,0.97,520.69
9,9,21848,21303,545,46.61,3.74,0.98,468.73


In [11]:
airport_gap = pd.read_csv(
    TABLES_DIR / "b3_airport_supply_gap.csv"
)

print("Columns:")
print(airport_gap.columns.tolist())

airport_gap.head(15)

Columns:
['hour', 'requests', 'fulfilled_requests', 'unfulfilled_requests', 'online_captains', 'avg_eta_min', 'fulfilment_rate', 'requests_per_online_captain', 'benchmark_fulfilment_rate', 'target_fulfilled_requests', 'additional_requests_to_fulfill', 'estimated_additional_captains']


,hour,requests,fulfilled_requests,unfulfilled_requests,online_captains,avg_eta_min,fulfilment_rate,requests_per_online_captain,benchmark_fulfilment_rate,target_fulfilled_requests,additional_requests_to_fulfill,estimated_additional_captains
0,22,11188,3146,8042,1589,9.90,0.28,7.04,0.95,"10,681.44","7,535.44","1,070.24"
1,23,13007,3413,9594,1533,10.04,0.26,8.48,0.95,"12,418.08","9,005.08","1,061.33"
2,1,11288,3143,8145,1567,9.83,0.28,7.20,0.95,"10,776.91","7,633.91","1,059.74"
3,2,9500,3073,6427,1612,9.58,0.32,5.89,0.95,"9,069.86","5,996.86","1,017.57"
4,0,9124,2933,6191,1537,9.54,0.32,5.94,0.95,"8,710.89","5,777.89",973.32
5,21,7293,2946,4347,1628,8.77,0.40,4.48,0.95,"6,962.79","4,016.79",896.66
6,3,6190,2916,3274,1613,8.01,0.47,3.84,0.95,"5,909.73","2,993.73",780.11
7,6,6776,4832,1944,2665,5.88,0.71,2.54,0.95,"6,469.20","1,637.20",643.91
8,4,4718,3219,1499,1828,6.18,0.68,2.58,0.95,"4,504.38","1,285.38",498.02
9,5,5291,3839,1452,2145,5.83,0.73,2.47,0.95,"5,051.44","1,212.44",491.53


In [12]:
if {"hour", "fulfillment_rate"}.issubset(airport_hourly.columns):

    hourly_plot = airport_hourly.sort_values("hour")

    plt.figure(figsize=(11, 5))

    plt.plot(
        hourly_plot["hour"],
        hourly_plot["fulfillment_rate"],
        marker="o"
    )

    plt.title("Airport Fulfillment Rate by Hour")
    plt.xlabel("Hour of Day")
    plt.ylabel("Fulfillment Rate")
    plt.xticks(range(24))
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

else:
    print("The hourly table does not contain the expected columns.")

The hourly table does not contain the expected columns.


### B1/B2 takeaway

Airport supply pressure is concentrated in the late-night period, particularly around 21:00–03:00.

Late-night airport trips also have a higher cancellation rate and a lower return-fare-within-20-min rate than the overall airport-trip population.

This suggests that the supply issue is not evenly distributed across the day and should be addressed with targeted interventions rather than broad captain acquisition.

In [13]:
trip_overall_path = TABLES_DIR / "b2_airport_trip_overall.csv"
trip_hourly_path = TABLES_DIR / "b2_airport_trip_hourly.csv"

trip_overall = pd.read_csv(trip_overall_path)
trip_hourly = pd.read_csv(trip_hourly_path)

print("Overall airport trip behavior")
display(trip_overall)

print("Hourly airport trip behavior")
display(trip_hourly.head(24))

Overall airport trip behavior


,trip_count,cancellation_rate,return_fare_rate,avg_fare_inr,avg_distance_km,avg_fare_per_km
0,60000,0.14,0.36,279.35,17.88,15.62


Hourly airport trip behavior


,hour,trip_count,cancellation_rate,return_fare_rate,avg_fare_inr,avg_distance_km,avg_fare_per_km
0,0,2503,0.17,0.28,279.88,17.86,15.67
1,1,2451,0.20,0.27,279.26,17.89,15.61
2,2,2616,0.20,0.25,277.80,17.79,15.62
3,3,2393,0.18,0.26,279.94,17.90,15.64
4,4,2539,0.12,0.39,277.32,17.66,15.70
5,5,2517,0.13,0.37,281.33,18.01,15.62
6,6,2494,0.11,0.38,279.56,17.88,15.64
7,7,2569,0.12,0.40,276.74,17.74,15.60
8,8,2545,0.12,0.38,278.88,17.87,15.60
9,9,2533,0.12,0.38,281.03,18.02,15.60


### B2 takeaway

Overall airport trips have:

- 13.77% cancellation rate
- 35.96% return fare within 20 minutes

For the late-night 21:00–03:00 window:

- Cancellation rate is approximately 17.09%
- Return fare within 20 minutes is approximately 29.84%

The late-night period therefore combines weaker fulfillment with higher cancellation and lower short-window return-trip behavior.

## 8. B3 — Should Rapido acquire more airport captains?

The analysis does not recommend immediately acquiring a large number of captains.

The preferred sequence is:

1. Target the late-night airport shortage.
2. Use incentives and repositioning to improve existing supply.
3. Measure whether fulfillment improves.
4. If the gap persists, acquire captains specifically for the affected airport/time/vehicle segments.

This reduces the risk of adding supply during periods when it is not needed.

In [14]:
b3_path = TABLES_DIR / "b3_airport_recommendation.csv"

b3_recommendation = pd.read_csv(b3_path)

b3_recommendation

,recommendation,reason,top_gap_hours,estimated_total_gap_captains,late_night_gap_share,overall_return_fare_rate,late_night_return_fare_rate,overall_cancellation_rate,late_night_cancellation_rate,measurement
0,Prioritize airport incentives and repositionin...,The majority of the estimated airport supply g...,"22,23,1,2,0,21","9,429.68",0.73,0.36,0.30,0.14,0.17,Run a controlled airport experiment comparing ...


## 9. Decision framework

The final recommendation is based on productive supply rather than signup volume alone.

For onboarding interventions:

Recoverable captains
× downstream approval probability
× approval-to-first-order rate
= expected incremental first orders

The current scenario assumes a 20% recovery rate and uses the observed 38.3% approval-to-first-order rate.

For airport supply, the priority is:

Demand gap
→ existing-supply intervention
→ controlled measurement
→ targeted acquisition if the gap remains

## 10. Final recommendations

### 1. Prioritize Permit completion

Focus on high-leakage Auto/Cab segments, especially Pune and Hyderabad.

A 20% recovery scenario corresponds to approximately 397 incremental Permit-stage approvals per month across the analyzed mature segment-month population.

### 2. Validate CAMP_WA_002

The campaign shows a +16.7 pp adjusted association with approval, but a randomized holdout is required before treating this as causal impact.

### 3. Improve Approved → First Order

2,596 approved captains do not complete a first order. Improving activation can create productive supply without acquiring another captain.

### 4. Target airport supply interventions

Focus on the 21:00–03:00 shortage window using incentives and repositioning before investing in targeted acquisition.

## 11. Limitations

- The data is synthetic and represents the assignment's supplied dataset.
- Intervention impact is scenario-based and not a causal estimate.
- CAMP_WA_002 exposure was not randomized.
- First-order completion is used as the available proxy for productive captain activation.
- The airport supply gap should not be interpreted as a direct requirement to acquire that exact number of captains.
- Long-term captain productivity, retention, and contribution margin are not available in the supplied data.